# Case-Study Modalities Trajectories

This notebook shows how different modality combinations predict individualized risk trajectories for randomly selected patients.

- **Case-study approach**: Visualizes specific patient examples to demonstrate model interpretability
- **Modality comparison**: Shows how clinical, omic, and imaging data contribute to predictions
- **Risk trajectories**: Displays personalized survival/risk curves over time
- **Uncertainty quantification**: Includes confidence intervals from multiple model samples

For each randomly selected patient:
1. Load trained models for all modality combinations
2. Sample multiple survival curves from each model 
3. Plot average survival curves with 5th-95th percentile confidence bands
4. Compare predictions across all modality combinations in a single plot5.

In [46]:
import os
import re
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from lifelines import KaplanMeierFitter, AalenJohansenFitter

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from survival_analysis.multimodal_samvae import Multimodal_SAMVAE
from survival_analysis.multimodal_samvae_cr import Multimodal_SAMVAE_cr
from data import split_cv_data_multimodal
from utils import load_datasets


## 2. Configuration

- **Dataset**: 'brca' or 'lgg'
- **Analysis Type**: Competing Risks (True) or Survival Analysis (False)
- **Modalities**: List of modality combinations to compare

In [47]:
SCRIPT_CONFIG = {
    # Dataset and analysis type
    "dataset": "brca",  # "brca" or "lgg"
    "competing_risks": False,  # True for competing risks, False for survival analysis
    # Model paths
    "results_base_dir": os.path.join("..", "results", "Final_Combinations"),
    # Modality combinations to compare (list of modality folder names)
    "modalities": [
        "clinical",
        "clinical_omic_cnv_omic_RNAseq",
        "omic_cnv_omic_RNAseq",
        "clinical_omic_cnv_omic_RNAseq_wsi_patches_10_patch",
        "clinical_wsi_patches_10_patch",
        "omic_cnv_omic_RNAseq_wsi_patches_10_patch",
        "wsi_patches_10_patch",
    ],
    # Model configuration (preferred, but will search for alternatives if not found)
    "latent_hidden": "[10]_[100]",  
    "seed": 0,
    "fold": 0,
    "n_folds": 5,    
    "batch_size": 1,  
    # Patient selection
    "patient_indices": None,  # None for random selection
    "max_patients": 5,  # Maximum number of patients to plot  
    "random_selection": True, 
    "random_seed": 42,    
    "num_samples": 100,  # Number of survival curves to sample per patient 
    "output_dir": os.path.join("figs", "case_study_modality_trajectories"),
    # Time parameters
    "time_mode": "weibull",
    "time_distribution": "Weibull",
    "time_hidden_size": 200,
    "normalization_loss": True,
}

## 3. Helper Functions

In [48]:
def get_model_path(base_dir, analysis_type, dataset, modality, n_folds, batch_size, latent_hidden, seed, fold):
    modality_path = os.path.join(base_dir, analysis_type, dataset, modality)
    if not os.path.exists(modality_path):
        return None
    target_file = f"model_fold_{fold}.pickle"
    fallback = None
    for root, _, files in os.walk(modality_path):
        if target_file in files and "seed_" in root:
            full = os.path.join(root, target_file)
            if f"seed_{seed}" in root:
                return full
            if fallback is None:
                fallback = full
    return fallback


def load_model(model_path, competing_risks):
    if not model_path or not os.path.exists(model_path):
        return None, None
    with open(model_path, "rb") as f:
        model_params = pickle.load(f)
    model_params.update({
        "image_resolution": 128,
        "final_image_resolution": 16,
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    })
    model = Multimodal_SAMVAE_cr(model_params) if competing_risks else Multimodal_SAMVAE(model_params)
    pt_path = model_path.replace(".pickle", ".pt")
    if not os.path.exists(pt_path):
        return None, None
    model.load_state_dict(torch.load(pt_path, map_location=model_params["device"]))
    model.eval()
    return model, model_params


def sample_survival_curves(model, patient_data, times_np, num_samples=100, competing_risks=False):
    with torch.no_grad():
        if competing_risks:
            num_risks = len(model(patient_data)["time_params"])
            results = {}
            for risk_idx in range(num_risks):
                surv_curves = []
                for _ in range(num_samples):
                    out_sample = model(patient_data)
                    alpha = out_sample["time_params"][risk_idx][0, 0].item()
                    lam = out_sample["time_params"][risk_idx][0, 1].item() / 365.25
                    surv_curves.append(np.exp(-np.power(times_np / lam, alpha)))
                surv_curves = np.stack(surv_curves)
                results[f"risk_{risk_idx}"] = {
                    "mean": np.mean(surv_curves, axis=0),
                    "median": np.median(surv_curves, axis=0),
                    "perc_5": np.percentile(surv_curves, 5, axis=0),
                    "perc_95": np.percentile(surv_curves, 95, axis=0),
                }
            return results
        surv_curves = []
        for _ in range(num_samples):
            out_sample = model(patient_data)
            alpha = out_sample["time_params"][0][0, 0].item()
            lam = out_sample["time_params"][0][0, 1].item() / 365.25
            surv_curves.append(np.exp(-np.power(times_np / lam, alpha)))
        surv_curves = np.stack(surv_curves)
        return {
            "mean": np.mean(surv_curves, axis=0),
            "median": np.median(surv_curves, axis=0),
            "perc_5": np.percentile(surv_curves, 5, axis=0),
            "perc_95": np.percentile(surv_curves, 95, axis=0),
        }


def load_patient_data_for_modality(dataset_name, modality, fold, config, n_wsi=None):
    args = {
        "datasets": [],
        "modes": [],
        "time_event": [],
        "n_folds": fold + 1,
        "time_distribution": ("weibull", 2) if config["time_distribution"] == "Weibull" else config["time_distribution"],
        "clinical_datasets": [],
        "omic_datasets": [],
        "wsi_datasets": [],
    }
    if "clinical" in modality:
        base_dataset = f"{dataset_name}_clinical" + ("_cr" if config["competing_risks"] else "")
        args["datasets"].append(base_dataset)
        args["modes"].append("clinical")
        args["clinical_datasets"].append(base_dataset)
    if "omic" in modality:
        def add_omic(name):
            args["datasets"].append(name)
            args["modes"].append("omic")
            args["omic_datasets"].append(name)
        if "adn" in modality:
            add_omic(f"{dataset_name}_adn")
        if "RNAseq" in modality and "cnv" not in modality and "adn" not in modality:
            add_omic(f"{dataset_name}_RNAseq")
        if "cnv" in modality:
            add_omic(f"{dataset_name}_cnv")
        if "RNAseq" in modality and "cnv" in modality:
            add_omic(f"{dataset_name}_RNAseq")
        if "miRNA" in modality:
            add_omic(f"{dataset_name}_miRNA")
    if "wsi" in modality:
        wsi_dataset = f"{dataset_name}_patches"
        args["datasets"].append(wsi_dataset)
        args["modes"].append("wsi")
        args["wsi_datasets"].append(wsi_dataset)
        args["N_wsi"] = n_wsi if n_wsi is not None else 1000
    time_event_name = f"{dataset_name}_time_event" + ("_cr" if config["competing_risks"] else "")
    args["time_event"] = [time_event_name]
    args["datasets"].append(time_event_name)
    args["modes"].append("time")
    args["clinical_input_dir"] = os.path.join(parent_dir, "data_preprocessing", "data", "clinical_data")
    args["omic_input_dir"] = os.path.join(parent_dir, "data_preprocessing", "data", "omic_data")
    args["wsi_input_dir"] = "/hdd/alba/samvae-main/data_preprocessing/data/wsi_data"
    args["time_event_input_dir"] = os.path.join(parent_dir, "data_preprocessing", "data", "time_event")
    try:
        all_tensors = load_datasets(args)
        if "wsi" in modality and n_wsi is not None:
            wsi_idx = len(args["clinical_datasets"]) + len(args["omic_datasets"])
            if wsi_idx < len(all_tensors) - 1:
                wsi_tensor = all_tensors[wsi_idx]
                current_patches = wsi_tensor.shape[1]
                if current_patches < n_wsi:
                    repeat_factor = (n_wsi + current_patches - 1) // current_patches
                    replicated = wsi_tensor.repeat(1, repeat_factor, 1, 1, 1)
                    all_tensors[wsi_idx] = replicated[:, :n_wsi, ...]
                elif current_patches > n_wsi:
                    all_tensors[wsi_idx] = wsi_tensor[:, :n_wsi, ...]
        cv_data_multimodal, _ = split_cv_data_multimodal(
            all_tensors, args["n_folds"], time_dist=args["time_distribution"]
        )
        data = [cv_data_multimodal[dataset_idx][fold] for dataset_idx in range(len(cv_data_multimodal))]
        return data
    except Exception:
        return None


def format_modality_name(modality):
    if modality == "clinical":
        return "Clinical"
    parts = modality.split("_")
    formatted_parts = []
    i = 0
    while i < len(parts):
        if parts[i] == "clinical":
            formatted_parts.append("Clinical")
            i += 1
        elif parts[i] == "omic":
            if i + 1 < len(parts):
                omic_type = parts[i + 1]
                if omic_type == "RNAseq":
                    formatted_parts.append("RNAseq")
                elif omic_type == "adn":
                    formatted_parts.append("DNA")
                elif omic_type == "cnv":
                    formatted_parts.append("CNV")
                elif omic_type == "miRNA":
                    formatted_parts.append("miRNA")
                else:
                    formatted_parts.append(omic_type)
                i += 2
            else:
                i += 1
        elif parts[i] == "wsi":
            if i + 3 < len(parts) and parts[i + 1] == "patches" and parts[i + 3] == "patch":
                formatted_parts.append(f"{parts[i + 2]} patches")
                i += 4
            else:
                i += 1
        else:
            i += 1
    return " + ".join(formatted_parts)


def compute_empirical_curve(data, times_np, competing_risks=False): 
    time_event_tensor = data[-1][2]
    all_times_years = time_event_tensor[:, 0].cpu().numpy() / 365.25
    all_events = time_event_tensor[:, 1].cpu().numpy().astype(int)
    if competing_risks:
        results = {}
        for risk_idx in range(2):
            aj = AalenJohansenFitter()
            try:
                aj.fit(durations=all_times_years, event_observed=all_events, event_of_interest=risk_idx + 1)
                def get_ci(t):
                    if t <= aj.cumulative_density_.index.max():
                        idx = aj.cumulative_density_.index <= t
                        cif_val = aj.cumulative_density_.loc[idx].iloc[-1].values[0]
                        ci_lower = aj.confidence_interval_.loc[idx, aj.confidence_interval_.columns[0]].iloc[-1]
                        ci_upper = aj.confidence_interval_.loc[idx, aj.confidence_interval_.columns[1]].iloc[-1]
                    else:
                        cif_val = aj.cumulative_density_.iloc[-1].values[0]
                        ci_lower = aj.confidence_interval_.iloc[-1, 0]
                        ci_upper = aj.confidence_interval_.iloc[-1, 1]
                    return 1.0 - cif_val, 1.0 - ci_upper, 1.0 - ci_lower
                cif_at_times, cif_lower, cif_upper = [], [], []
                for t in times_np:
                    v, lo, hi = get_ci(t)
                    cif_at_times.append(v)
                    cif_lower.append(lo)
                    cif_upper.append(hi)
                results[f"risk_{risk_idx}"] = {
                    "mean": np.array(cif_at_times),
                    "lower_ci": np.array(cif_lower),
                    "upper_ci": np.array(cif_upper),
                }
            except Exception:
                results[f"risk_{risk_idx}"] = {
                    "mean": np.ones_like(times_np),
                    "lower_ci": np.ones_like(times_np),
                    "upper_ci": np.ones_like(times_np),
                }
        return results
    event_observed = (all_events == 1).astype(int)
    kmf = KaplanMeierFitter()
    kmf.fit(durations=all_times_years, event_observed=event_observed)
    survival_at_times, survival_lower, survival_upper = [], [], []
    for t in times_np:
        if t <= kmf.survival_function_.index.max():
            idx = kmf.survival_function_.index <= t
            surv_val = kmf.survival_function_.loc[idx].iloc[-1].values[0]
            ci_lower = kmf.confidence_interval_.loc[idx, kmf.confidence_interval_.columns[0]].iloc[-1]
            ci_upper = kmf.confidence_interval_.loc[idx, kmf.confidence_interval_.columns[1]].iloc[-1]
        else:
            surv_val = kmf.survival_function_.iloc[-1].values[0]
            ci_lower = kmf.confidence_interval_.iloc[-1, 0]
            ci_upper = kmf.confidence_interval_.iloc[-1, 1]
        survival_at_times.append(surv_val)
        survival_lower.append(ci_lower)
        survival_upper.append(ci_upper)
    return {
        "mean": np.array(survival_at_times),
        "lower_ci": np.array(survival_lower),
        "upper_ci": np.array(survival_upper),
    }

## 4. Main Plotting Function

In [49]:
def plot_patient_model_comparison(config):
    os.makedirs(config["output_dir"], exist_ok=True)
    analysis_type = "Competing_Risks" if config["competing_risks"] else "Survival_Analysis"
    dataset = config["dataset"]
    modality_base_path = os.path.join(config["results_base_dir"], analysis_type, dataset)
    if os.path.exists(modality_base_path):
        available_modalities = [
            d for d in os.listdir(modality_base_path)
            if os.path.isdir(os.path.join(modality_base_path, d))
            and not d.endswith((".tex", ".txt", ".pkl"))
        ]
    else:
        available_modalities = config["modalities"]
    if not available_modalities:
        return
    times_np = np.linspace(0, 10, 200)
    data = load_patient_data_for_modality(dataset, available_modalities[0], config["fold"], config, n_wsi=None)
    if data is None:
        return
    num_patients = data[0][2].shape[0]
    if config["patient_indices"] is None:
        if config.get("random_selection", False):
            np.random.seed(config.get("random_seed", 42))
            max_patients = min(config["max_patients"], num_patients)
            patient_indices = sorted(np.random.choice(num_patients, max_patients, replace=False).tolist())
        else:
            patient_indices = list(range(min(config["max_patients"], num_patients)))
    else:
        patient_indices = [idx for idx in config["patient_indices"] if idx < num_patients]
    modality_color_map = {
        "clinical": "blue",
        "clinical_omic_cnv_omic_RNAseq_wsi_patches_10_patch": "green",
        "omic_cnv_omic_RNAseq": "orange",
        "clinical_omic_cnv_omic_RNAseq": "red",
        "omic_cnv_omic_RNAseq_wsi_patches_10_patch": "purple",
        "wsi_patches_10_patch": "cyan",
        "clinical_wsi_patches_10_patch": "magenta",
        "clinical_omic_adn": "red",
        "clinical_omic_adn_omic_cnv_omic_miRNA": "red",
        "omic_adn": "orange",
        "omic_adn_omic_cnv_omic_miRNA": "orange",
        "clinical_omic_adn_wsi_patches_10_patch": "green",
        "omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch": "purple",
        "clinical_wsi_patches_5_patch": "magenta",
        "wsi_patches_5_patch": "cyan",
    }
    palette = ["blue", "green", "orange", "red", "purple", "cyan", "magenta", "brown"]
    colors = [modality_color_map.get(m, palette[i % len(palette)]) for i, m in enumerate(available_modalities)]
    for patient_idx in patient_indices:
        if config["competing_risks"]:
            if all(
                os.path.exists(os.path.join(config["output_dir"], f"patient_{patient_idx}_risk_{r}_comparison.png"))
                for r in [0, 1]
            ):
                continue
        else:
            output_path = os.path.join(config["output_dir"], f"patient_{patient_idx}_model_comparison.png")
            if os.path.exists(output_path):
                continue
        empirical_curve = None
        if config["competing_risks"]:
            risk_data = {}
        else:
            fig, ax = plt.subplots(figsize=(14, 8))
            models_plotted = 0
        for modality_idx, modality in enumerate(available_modalities):
            model_path = get_model_path(
                config["results_base_dir"], analysis_type, dataset, modality,
                config["n_folds"], config["batch_size"], config["latent_hidden"],
                config["seed"], config["fold"]
            )
            model, model_params = load_model(model_path, config["competing_risks"])
            if model is None:
                continue
            n_wsi = None
            if "wsi" in modality:
                match = re.search(r"patches_(\d+)_patch", modality)
                if match:
                    n_wsi = int(match.group(1))
                else:
                    n_wsi = model_params.get("N_wsi", model_params.get("n_wsi", 1000))
            data = load_patient_data_for_modality(dataset, modality, config["fold"], config, n_wsi=n_wsi)
            if data is None:
                continue
            if empirical_curve is None:
                empirical_curve = compute_empirical_curve(data, times_np, config["competing_risks"])
            patient_data = [x[2][patient_idx:patient_idx+1].to(model.device) for x in data[:-1]]
            try:
                curves = sample_survival_curves(model, patient_data, times_np, config["num_samples"], config["competing_risks"])
            except Exception:
                continue
            if config["competing_risks"] and isinstance(curves, dict) and "risk_0" in curves:
                modality_label = format_modality_name(modality)
                color = colors[modality_idx]
                for risk_key in sorted(curves.keys()):
                    risk_idx = int(risk_key.split("_")[1])
                    risk_data.setdefault(risk_idx, []).append((modality_label, curves[risk_key], color))
            else:
                if models_plotted == 0 and empirical_curve is not None:
                    ax.plot(times_np, empirical_curve["mean"], color="gray", linewidth=3,
                            label="Kaplan-Meier", alpha=0.5, linestyle="--", zorder=1)
                    ax.fill_between(times_np, empirical_curve["lower_ci"], empirical_curve["upper_ci"],
                                    color="gray", alpha=0.15, zorder=1)
                color = colors[modality_idx]
                label = format_modality_name(modality)
                ax.plot(times_np, curves["mean"], color=color, linewidth=2.5, label=label, alpha=0.9, zorder=2)
                ax.fill_between(times_np, curves["perc_5"], curves["perc_95"], color=color, alpha=0.15, zorder=2)
                models_plotted += 1
        if config["competing_risks"]:
            if not risk_data:
                continue
            for risk_idx in sorted(risk_data.keys()):
                fig, ax = plt.subplots(figsize=(14, 8))
                if empirical_curve is not None and f"risk_{risk_idx}" in empirical_curve:
                    risk_curve = empirical_curve[f"risk_{risk_idx}"]
                    ax.plot(times_np, risk_curve["mean"], color="gray", linewidth=3,
                            label="Aalen-Johansen", alpha=0.5, linestyle="--", zorder=1)
                    ax.fill_between(times_np, risk_curve["lower_ci"], risk_curve["upper_ci"],
                                    color="gray", alpha=0.15, zorder=1)
                for modality_label, risk_curves, color in risk_data[risk_idx]:
                    ax.plot(times_np, risk_curves["mean"], color=color, linewidth=2.5,
                            label=modality_label, alpha=0.9, zorder=2)
                    ax.fill_between(times_np, risk_curves["perc_5"], risk_curves["perc_95"],
                                    color=color, alpha=0.15, zorder=2)
                ax.set_xlim(0, times_np[-1])
                ax.set_ylim(0, 1.05)
                ax.set_xlabel("Time (years)", fontsize=16)
                ax.set_ylabel("Survival Probability (1-CIF)", fontsize=16)
                ax.set_title(f"{dataset.upper()}-CR - Patient {patient_idx}: Risk {risk_idx + 1} Individual Curves",
                             fontsize=20, pad=20)
                ax.legend(loc="best", fontsize=15, framealpha=0.9)
                ax.grid(True, alpha=0.3, linestyle="--")
                ax.tick_params(labelsize=12)
                plt.tight_layout()
                output_path = os.path.join(config["output_dir"], f"patient_{patient_idx}_risk_{risk_idx}_comparison.png")
                plt.savefig(output_path, dpi=300, bbox_inches="tight")
                plt.close()
        else:
            if models_plotted == 0:
                plt.close(fig)
                continue
            ax.set_xlim(0, times_np[-1])
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("Time (years)", fontsize=16)
            ax.set_ylabel("Survival Probability", fontsize=16)
            ax.set_title(f"{dataset.upper()}-SA - Patient {patient_idx}: Individual Multimodal Curves",
                         fontsize=20, pad=20)
            ax.legend(loc="best", fontsize=15, framealpha=0.9)
            ax.grid(True, alpha=0.3, linestyle="--")
            ax.tick_params(labelsize=12)
            plt.tight_layout()
            output_path = os.path.join(config["output_dir"], f"patient_{patient_idx}_model_comparison.png")
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            plt.close()

## 5. Plots

In [ ]:
datasets = ["brca", "lgg"]
analysis_types = [("Competing Risks", True), ("Survival Analysis", False)]

for dataset in datasets:
    for analysis_name, competing_risks in analysis_types:
        config = SCRIPT_CONFIG.copy()
        config["dataset"] = dataset
        config["competing_risks"] = competing_risks
        config["max_patients"] = 1
        config["output_dir"] = os.path.join(
            "figs", "case_study_modality_trajectories", dataset, analysis_name.replace(" ", "_")
        )
        try:
            plot_patient_model_comparison(config)
        except Exception as e:
            print(f"Error processing {dataset} - {analysis_name}: {e}")
            continue